# Where the mistakes are

Research only: read held-out predictions saved by `02_train_and_validate.ipynb`
and compare scores with human annotations. Examine errors by well, tracker
context, and score confidence. This notebook does not set inference decisions
or supply an expected male count.

Run from `notebooks/` with trusted `../data/labeled_crops.npz` and
`../data/cv_predictions.npz`. Tracker context also needs the matching session
folders in `../data/sessions/`, including the original still filenames and
FlyTracker CSVs. These inputs are not included in Git. The NumPy files contain
object arrays and are loaded with `allow_pickle=True`.

Plots and review CSVs are written to `../results/research-audit/`.


## Imports and configuration

In [ ]:
# Import dependencies
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
# Look for a file in the shared data folder first, then in the working directory
def find_data_file(name):
    # Running from notebooks/ locally, or from an upload on Colab
    for candidate in (Path("../data") / name, Path(name)):
        if candidate.exists():
            return candidate

    # Report missing inputs
    raise ValueError(f"Missing {name}; put it in ../data or the current directory")

In [ ]:
# Point at the dataset, the held-out predictions, and the session folders
DATASET_PATH = find_data_file("labeled_crops.npz")
PREDICTIONS_PATH = find_data_file("cv_predictions.npz")
SESSION_ROOT = find_data_file(name="sessions")
OUTPUT_DIRECTORY = Path("../results/research-audit")

# Map workbook tab prefixes to session folder names
SESSION_FOLDERS = {
    "0714": "07142026",
    "0717": "07172026",
    "0721": "7212026",
}

# Call a fly male above this score, and call the score confident outside the band
DECISION_THRESHOLD = 0.5
CONFIDENT_LOW = 0.1
CONFIDENT_HIGH = 0.9

# Define what counts as an edge case in the tracker's eyes
CONTACT_DISTANCE = 50.0
WALL_DISTANCE = 2.0
MISALIGNMENT_LIMIT = 30.0

# Define the number of columns in review sheets
REVIEW_COLUMNS = 8

# Define how still frame numbers map to tracker rows
FRAME_OFFSET = 1
MINIMUM_FILE_SIZE = 1_500_000

## Function definitions

In [ ]:
# Find one well's FlyTracker folder, however deeply the session nests it
def find_tracker_directory(session_directory, well_number):
    # Accept both the flat layout and the arena_N sub-folder layout
    matches = sorted(session_directory.rglob(f"*_arena_{well_number}-trackfeat.csv"))
    matches = [path for path in matches if path.is_dir()]
    if len(matches) != 1:
        raise ValueError(f"Expected one trackfeat folder for well {well_number}, found {len(matches)}")

    # Return the matching folder
    return matches[0]

In [ ]:
# Find the frame number of the first usable still in a session
def first_still_frame(session_directory):
    # Look in the session folder and in a photos sub-folder
    paths = list(session_directory.glob("frame_*.png")) + list((session_directory / "photos").glob("frame_*.png"))
    frames = [
        int(re.search(r"frame_(\d+)", path.name).group(1))
        for path in paths
        if path.stat().st_size >= MINIMUM_FILE_SIZE
    ]
    if not frames:
        raise ValueError(f"No usable stills in {session_directory}")

    # Return the smallest frame number
    return min(frames)

In [ ]:
# Measure how far a crop's dark blob leans away from horizontal
def crop_misalignment(crop):
    # Keep only the darkest pixels
    darkness = 255.0 - crop.astype(np.float32)
    darkness = np.where(darkness > np.percentile(darkness, 85), darkness, 0.0)
    weight = darkness.sum()
    if weight < 1:
        return np.nan

    # Take the principal axis of the dark pixels
    rows, columns = np.mgrid[: crop.shape[0], : crop.shape[1]]
    centre_x = (columns * darkness).sum() / weight
    centre_y = (rows * darkness).sum() / weight
    xx = ((columns - centre_x) ** 2 * darkness).sum() / weight
    yy = ((rows - centre_y) ** 2 * darkness).sum() / weight
    xy = ((columns - centre_x) * (rows - centre_y) * darkness).sum() / weight

    # Return the angle in degrees
    return abs(0.5 * np.degrees(np.arctan2(2 * xy, xx - yy)))

In [ ]:
# Pull the tracker's view of every labelled crop
def read_tracker_context(table, crops):
    # Cache the per-well tracker tables
    cache = {}
    records = []
    for row in table.itertuples():
        key = (row.well, row.arena)
        if key not in cache:
            session_directory = SESSION_ROOT / SESSION_FOLDERS[row.well[:4]]
            tracker_directory = find_tracker_directory(session_directory, row.arena)
            tracks = [pd.read_csv(tracker_directory / f"fly{index}.csv") for index in range(1, 6)]
            cache[key] = (tracks, first_still_frame(session_directory))
        tracks, first_frame = cache[key]

        # Read this fly's row and its distance to the nearest other fly
        tracker_row = row.frame - first_frame + FRAME_OFFSET
        if not 0 <= tracker_row < min(len(track) for track in tracks):
            raise ValueError(f"Tracker row {tracker_row} outside the tables for {row.well}")
        own = tracks[row.fly - 1]
        positions_x = np.array([track["pos x"].iloc[tracker_row] for track in tracks])
        positions_y = np.array([track["pos y"].iloc[tracker_row] for track in tracks])
        separations = np.hypot(positions_x - positions_x[row.fly - 1], positions_y - positions_y[row.fly - 1])
        separations[row.fly - 1] = np.inf

        records.append(
            {
                "dist_to_wall": float(own["dist_to_wall"].iloc[tracker_row]),
                "min_fg_dist": float(own["min fg dist"].iloc[tracker_row]),
                "nearest_px": float(np.nanmin(separations)),
                "misalignment": crop_misalignment(crops[row.Index]),
            }
        )

    # Return one row per crop
    return pd.DataFrame(records, index=table.index)

In [ ]:
# Lay crops out in a captioned grid
def draw_sheet(crops, captions, title, output_path, columns=REVIEW_COLUMNS):
    # Skip empty selections
    if len(crops) == 0:
        print(f"{title}: nothing to show")
        return

    # Work out the grid shape
    rows = int(np.ceil(len(crops) / columns))
    figure, axes = plt.subplots(rows, columns, figsize=(columns * 1.5, rows * 1.8))
    axes = np.atleast_2d(axes).reshape(rows, columns)
    for position in range(rows * columns):
        axis = axes[position // columns, position % columns]
        axis.set_xticks([])
        axis.set_yticks([])
        if position >= len(crops):
            axis.axis("off")
            continue
        axis.imshow(crops[position], cmap="gray", vmin=0, vmax=255, interpolation="nearest")
        axis.set_xlabel(captions[position], fontsize=8)

    # Save the sheet
    figure.suptitle(title, fontsize=11)
    figure.tight_layout()
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure.savefig(output_path, dpi=140)
    plt.show()
    plt.close(figure)

## Load the predictions

In [ ]:
# Join the dataset with its held-out predictions
dataset = np.load(DATASET_PATH, allow_pickle=True)
predictions = np.load(PREDICTIONS_PATH, allow_pickle=True)
crops = dataset["crops"]

# Validate the dataset and prediction alignment
if not np.array_equal(dataset["meta"], predictions["meta"]):
    raise ValueError("Prediction provenance does not match the dataset")
if not np.array_equal(dataset["y"], predictions["labels"]):
    raise ValueError("Prediction labels do not match the dataset")
probabilities = np.asarray(predictions["probabilities"], dtype=np.float32)
if probabilities.shape != (len(crops),) or not np.isfinite(probabilities).all():
    raise ValueError("Expected one finite held-out score for every crop")
if not ((probabilities >= 0) & (probabilities <= 1)).all():
    raise ValueError("Held-out scores must fall between 0 and 1")

table = pd.DataFrame(
    {
        "well": [row[0] for row in dataset["meta"]],
        "arena": [int(row[1]) for row in dataset["meta"]],
        "frame": [int(row[2]) for row in dataset["meta"]],
        "fly": [int(row[3]) for row in dataset["meta"]],
        "label": dataset["y"].astype(int),
        "probability": predictions["probabilities"],
        "date": predictions["dates"],
    }
)

# Call every fly from its own score
table["call"] = (table["probability"] > DECISION_THRESHOLD).astype(int)
table["error"] = table["call"] != table["label"]
table["confident"] = (table["probability"] < CONFIDENT_LOW) | (table["probability"] > CONFIDENT_HIGH)
table["confidence"] = (table["probability"] - 0.5).abs()

missed_males = int(((table["label"] == 1) & (table["call"] == 0)).sum())
false_males = int(((table["label"] == 0) & (table["call"] == 1)).sum())
print(f"{len(table)} crops | {int(table['label'].sum())} male | errors {int(table['error'].sum())} ({table['error'].mean():.1%})")
print(f"Males called female: {missed_males} | females called male: {false_males}")
print(f"Confident on {table['confident'].mean():.1%} of crops, {table['error'][table['confident']].mean():.1%} wrong there "
      f"against {table['error'][~table['confident']].mean():.1%} elsewhere")

## Which wells the errors come from

Separate missed males from females called male, using the human annotations as
the evaluation reference. Differences between wells may reflect image conditions,
annotation disagreement, or model limitations and need review.


In [ ]:
# Break the errors down by well and by direction
summary = []
for well, group in table.groupby("well"):
    males = group[group["label"] == 1]
    females = group[group["label"] == 0]
    summary.append(
        {
            "well": well,
            "crops": len(group),
            "error rate": group["error"].mean(),
            "males": len(males),
            "males missed": float((males["call"] == 0).mean()) if len(males) else np.nan,
            "females called male": float((females["call"] == 1).mean()) if len(females) else np.nan,
            "confident share": group["confident"].mean(),
        }
    )
print(pd.DataFrame(summary).set_index("well").to_string(float_format=lambda value: f"{value:.1%}"))

## Can tracker context help identify errors

Compare errors around close contact, the arena wall, and crop misalignment.
Tracker distances and a principal-axis estimate from the crop provide screening
features; these flags are not ground truth about the cause of a disagreement.


In [ ]:
# Flag the edge cases the tracker can describe
context = read_tracker_context(table, crops)
table = table.join(context)

flags = {
    "touching another fly": (table["min_fg_dist"] <= 0) | (table["nearest_px"] < CONTACT_DISTANCE),
    "at the wall": table["dist_to_wall"] < WALL_DISTANCE,
    "crop misaligned": table["misalignment"] > MISALIGNMENT_LIMIT,
}

print(f"{'edge case':24s} {'crops':>6s} {'error rate':>11s} {'elsewhere':>10s}")
any_flag = np.zeros(len(table), dtype=bool)
for name, flag in flags.items():
    flag = flag.fillna(False).to_numpy()
    any_flag |= flag
    print(f"{name:24s} {int(flag.sum()):6d} {table['error'][flag].mean():11.1%} {table['error'][~flag].mean():10.1%}")
print(f"{'any of the three':24s} {int(any_flag.sum()):6d} {table['error'][any_flag].mean():11.1%} {table['error'][~any_flag].mean():10.1%}")
table["flagged"] = any_flag

## How far confidence gets you

Measure error rates across score bands and the accuracy retained by several
confidence thresholds. Report coverage alongside accuracy so filtering does not
hide how many crops remain uncertain.


In [ ]:
# Error rate by confidence band, then the fixed band the pipeline applies
for low, high in [(0.0, 0.1), (0.1, 0.25), (0.25, 0.4), (0.4, 0.5)]:
    selected = (table["confidence"] >= low) & (table["confidence"] <= high)
    print(f"|p - 0.5| in [{low:.2f}, {high:.2f}] | {int(selected.sum()):4d} crops | error rate {table['error'][selected].mean():.1%}")

print()
print("Fixed confidence band, evaluated one held-out date at a time:")
dates = sorted(table["date"].unique())
for low, high in [(0.2, 0.8), (0.1, 0.9), (0.05, 0.95)]:
    kept = (table["probability"] < low) | (table["probability"] > high)
    shares = [kept[table["date"] == date].mean() for date in dates]
    accuracies = [
        (table["call"][kept & (table["date"] == date)] == table["label"][kept & (table["date"] == date)]).mean()
        for date in dates
    ]
    print(f"  outside {low} to {high} | keeps {np.mean(shares):.1%} | accuracy {np.mean(accuracies):.3f} "
          f"({' / '.join(f'{value:.3f}' for value in accuracies)})")

## Look at the mistakes

`M to F` means the human annotation is male and the score is at or below 0.5.
`F to M` is the reverse. Captions show the score, and the highest-confidence
disagreements appear first. Every binary call comes directly from the crop score.


In [ ]:
# Describe one crop for a caption
def caption_for(row):
    label = "M" if row.label else "F"
    call = "M" if row.call else "F"
    return f"{row.well[-2:]} {label} to {call} {row.probability:.2f}"


# Draw the mistakes in each direction and the confident ones
selections = {
    "missed_males": ((table["label"] == 1) & (table["call"] == 0), "Males the model called female"),
    "false_males": ((table["label"] == 0) & (table["call"] == 1), "Females the model called male"),
    "confident_mistakes": (table["confident"] & table["error"], "Confident and wrong"),
}
for name, (selected, title) in selections.items():
    chosen = table[selected].sort_values("confidence", ascending=False).head(40)
    draw_sheet(
        crops=[crops[index] for index in chosen.index],
        captions=[caption_for(row) for row in chosen.itertuples()],
        title=f"{title} ({int(selected.sum())} crops)",
        output_path=OUTPUT_DIRECTORY / f"{name}.png",
    )

## Export the crops worth re-checking

Export every disagreement between the score-based call and the human annotation.
The `your_call` column is blank for a fresh visual review. Review annotations are
for supervised dataset curation and evaluation only.


In [ ]:
# Write the review list and its contact sheets
review = table[table["error"]]
review = review.sort_values(["well", "frame", "fly"])

review_list = review[["well", "frame", "fly", "label", "call", "probability"]].copy()
review_list["annotator"] = np.where(review_list["label"] == 1, "M", "F")
review_list["model"] = np.where(review_list["probability"] > DECISION_THRESHOLD, "M", "F")
review_list["number"] = range(1, len(review_list) + 1)
review_list["your_call"] = ""
review_list = review_list[["number", "well", "frame", "fly", "annotator", "model", "probability", "your_call"]]
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
review_list.to_csv(OUTPUT_DIRECTORY / "review_list.csv", index=False)

# Number the crops so the sheet and the list line up, 40 to a page
for page_start in range(0, len(review), 40):
    page = review.iloc[page_start : page_start + 40]
    numbers = range(page_start + 1, page_start + 1 + len(page))
    draw_sheet(
        crops=[crops[index] for index in page.index],
        captions=[f"#{number}" for number in numbers],
        title=f"Review sheet, crops {page_start + 1} to {page_start + len(page)}",
        output_path=OUTPUT_DIRECTORY / f"review_sheet_{page_start // 40 + 1}.png",
    )

print(f"{len(review_list)} crops to review, list at {OUTPUT_DIRECTORY / 'review_list.csv'}")

## Compare annotations and scores by frame

Summarize the number of human-labelled males and the number of scores above 0.5
within each annotated snapshot. This is an evaluation summary of the labelled
crops present in the dataset. It does not estimate missing flies or impose a
male count on inference; unknown labels may have been omitted during extraction.


In [ ]:
# Summarise annotation and score calls within the evaluation set
comparison = []
for well, group in table.groupby("well"):
    per_frame = group.groupby("frame").agg(
        annotated=("label", "sum"),
        called=("call", "sum"),
        evaluated_crops=("label", "size"),
    )
    comparison.append(
        {
            "well": well,
            "frames": len(per_frame),
            "evaluated crops median": per_frame["evaluated_crops"].median(),
            "annotated median": per_frame["annotated"].median(),
            "score calls median": per_frame["called"].median(),
        }
    )
print(pd.DataFrame(comparison).set_index("well").to_string())


## Notes

These are held-out crop classification results against human annotations. They
measure performance on the represented recording dates, not uninterrupted identity
tracking or guaranteed correctness on new sessions. A confident disagreement can
come from either the annotation or the classifier; inspect the crop before changing
its training label. Tracker identifiers may switch between flies over time, so
pooling their scores does not establish a persistent biological identity.
